# Lab 1 — Spot market making, Bachelier model
Slides 6–19. Time in trading days: the horizon $\tau$ is one session, $S_t = S_0 + \sigma W_t$ with $\sigma$ the daily vol, quotes $B_t = S_t-\delta^b_t$, $A_t = S_t+\delta^a_t$, fills with intensity $\lambda e^{-\kappa\delta}$ ($\lambda=100$, $\kappa=1.5$ per day as on slide 9), inventory $Q_t = N^b_t - N^a_t$.

Objective (slide 11): maximize $J(\delta)=\mathbb E\big[\int_0^\tau \delta^b_t dN^b_t+\delta^a_t dN^a_t-\alpha\int_0^\tau |Q_t|^2dt-\beta|Q_\tau|^2\big]$.

In [ ]:
import math, torch, numpy as np, matplotlib.pyplot as plt
from omm import *
torch.manual_seed(0)

S0, sigma, tau, N = 100.0, 1.0, 1.0, 390      # daily vol 1.0 (~16% annualised), 1-minute steps
lam, kappa = 100.0, 1.5                        # order-flow intensity and elasticity (slide 8)
alpha, beta = 0.01, 0.05                       # running / terminal inventory penalties
dt = tau / N
print(DEV, "FAST" if FAST else "full scale")

## Simulation (slides 7–10)
`simulate` rolls out a quoting rule $\delta(t,S_t,Q_t)\in\mathbb R^2$ over one session and returns the sample objective of slide 11 together with the pieces DSO needs (`rbar`, `logp`, `cost`, `term`).

In [ ]:
def feats(t, S, Q):
    return torch.stack([t / tau * torch.ones_like(S), (S - S0) / (sigma * math.sqrt(tau)), Q / 10.0], -1)

def simulate(policy, B, seed=None):
    g = torch.Generator(device=DEV)
    if seed is not None: g.manual_seed(seed)
    dW = math.sqrt(dt) * torch.randn(B, N, device=DEV, generator=g)         # common random numbers for S
    S, Q = torch.full((B,), S0, device=DEV), torch.zeros(B, device=DEV)
    out = dict(rbar=[], logp=[], cost=[], d=[], dN=[], Q=[Q], S=[S])
    spread = inv = 0.0
    for k in range(N):
        d = policy(k * dt, S, Q)                                            # (B,2) bid / ask spreads
        lam_dt = lam * torch.exp(-kappa * d) * dt
        dN, lp = fills(lam_dt)                                              # Poisson fills at the posted quotes
        out["rbar"].append((d * lam_dt).sum(-1)); out["logp"].append(lp.sum(-1))
        out["cost"].append(alpha * Q ** 2 * dt); out["d"].append(d.detach()); out["dN"].append(dN)
        spread = spread + (d.detach() * dN).sum(-1)                         # spread capture
        inv = inv + Q * sigma * dW[:, k]                                    # int Q dS (zero mean)
        Q, S = Q + dN[:, 0] - dN[:, 1], S + sigma * dW[:, k]
        out["Q"].append(Q); out["S"].append(S)
    out = {k: torch.stack(v, 1) for k, v in out.items()}
    out.update(term=beta * Q ** 2, spread=spread, pnl=spread + inv)
    out["J"] = spread - out["cost"].sum(1) - out["term"]                    # sample objective (slide 11)
    return out

myopic = lambda t, S, Q: torch.full((len(S), 2), 1 / kappa, device=DEV)    # maximises spread capture alone

## DSO — Algorithm 2 (slide 12)
$\delta^\theta_t = \tfrac1\kappa + \Phi(t,S_t,Q^{\delta^\theta}_t;\theta)$. The law of the fills depends on $\theta$, so $\nabla_\theta \hat J$ is taken with the score-function estimator implemented in `dso_surrogate` (reward-to-go with a batch baseline); autograd of the surrogate is an unbiased estimate of $\nabla_\theta J(\delta^\theta)$.

In [ ]:
net = MLP(3, 2, zero_last=True).to(DEV)
dso_policy = lambda t, S, Q: 1 / kappa + ckpt(net, feats(t, S, Q))

def dso_step():
    o = simulate(dso_policy, B=scale(4096, 8))
    surr, J = dso_surrogate(o["rbar"], o["logp"], o["cost"], o["term"])
    return -surr, J

hist_dso = train(dso_step, net.parameters(), iters=scale(1500), lr=2e-3, clip=10.0, name="DSO")

## DGM — Algorithm 3 (slides 14–17)
$\mathcal L v = \partial_t v + \tfrac12\sigma^2\partial_{ss}v - \alpha q^2 + \tfrac\lambda\kappa\big(e^{\kappa\Delta^b_q v-1}+e^{\kappa\Delta^a_q v-1}\big)=0$, $\;v(\tau,s,q)=-\beta q^2$, with $\Delta^b_q v = v(t,s,q+1)-v(t,s,q)$, $\Delta^a_q v = v(t,s,q-1)-v(t,s,q)$. Spreads recovered as $\delta^{b}=\tfrac1\kappa-\Delta^b_q v$, $\delta^{a}=\tfrac1\kappa-\Delta^a_q v$ (slide 17).

In [ ]:
Qmax = 20
vnet = MLP(3, 1, depth=3).to(DEV)
V = lambda t, s, q: vnet(feats(t, s, q)).squeeze(-1)

def hjb(t, s, q):                                                        # L v^theta at collocation points
    t, s = t.requires_grad_(True), s.requires_grad_(True)
    v = V(t, s, q)
    v_t, v_ss = grad(v, t), grad(grad(v, s), s)
    Db, Da = V(t, s, q + 1) - v, V(t, s, q - 1) - v                    # post-trade value increments
    return v_t + 0.5 * sigma ** 2 * v_ss - alpha * q ** 2 + lam / kappa * (torch.exp(kappa * Db - 1) + torch.exp(kappa * Da - 1))

def dgm_step(B=scale(2048, 4), w=10.0):
    t, s = tau * torch.rand(B, device=DEV), S0 + 2 * sigma * math.sqrt(tau) * torch.randn(B, device=DEV)
    q = torch.randint(-Qmax, Qmax + 1, (B,), device=DEV).float()
    loss = hjb(t, s, q).pow(2).mean() + w * (V(torch.full_like(t, tau), s, q) + beta * q ** 2).pow(2).mean()
    return loss, loss.item()

hist_dgm = train(dgm_step, vnet.parameters(), iters=scale(4000), lr=2e-3, name="DGM")

def dgm_policy(t, S, Q):
    t = t * torch.ones_like(S); v = V(t, S, Q)
    return torch.stack([1 / kappa + v - V(t, S, Q + 1), 1 / kappa + v - V(t, S, Q - 1)], -1)

## Exact solution (reference)
Nothing in the objective depends on $s$, so $v=v(t,q)$ and the HJB is a system of ODEs indexed by $q\in\mathbb Z$. With $u_q = e^{\kappa v_q}$ it is linear, $\dot u = Mu$ with $M_{qq}=\alpha\kappa q^2$, $M_{q,q\pm1}=-\lambda/e$, so $u(t)=e^{-(\tau-t)M}u(\tau)$, $u_q(\tau)=e^{-\kappa\beta q^2}$ (inventory truncated at $\pm\bar Q$). Both DSO and DGM are checked against it.

In [ ]:
from scipy.linalg import expm
Qbar = 40; qs = np.arange(-Qbar, Qbar + 1); nq = len(qs)
M = np.diag(alpha * kappa * qs ** 2) - lam / math.e * (np.eye(nq, k=1) + np.eye(nq, k=-1))
u = np.stack([expm(-(tau - t) * M) @ np.exp(-kappa * beta * qs ** 2) for t in np.linspace(0, tau, N + 1)])
v_ex = np.log(u) / kappa                                                  # (N+1, nq)
db_ex = 1 / kappa + v_ex[:, :-1] - v_ex[:, 1:]                            # bid spread at q = qs[:-1]
da_ex = 1 / kappa + v_ex[:, 1:] - v_ex[:, :-1]                            # ask spread at q = qs[1:]
print(f"J* = v(0, q=0) = {v_ex[0, Qbar]:.4f}")

_db, _da = (torch.tensor(x, dtype=torch.float32, device=DEV) for x in (db_ex, da_ex))
def exact_policy(t, S, Q):
    k, j = int(round(t / dt)), (Q.long() + Qbar).clamp(1, nq - 2)
    return torch.stack([_db[k, j], _da[k, j - 1]], -1)

## Comparison
Backtest on common random numbers, then the learned spreads against the exact ones.

In [ ]:
policies = {"myopic 1/k": myopic, "DSO": dso_policy, "DGM": dgm_policy, "exact": exact_policy}
with torch.no_grad():
    for i, (name, pol) in enumerate(policies.items()):
        o = simulate(pol, B=scale(20000, 8), seed=1)
        report(name, o["pnl"], o["J"], o["Q"][:, -1], o["dN"].sum((1, 2)).mean().item(), header=(i == 0))

In [ ]:
qq = torch.arange(-15, 16, device=DEV).float(); Sq = torch.full_like(qq, S0); jj = np.arange(-15, 16) + Qbar
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
with torch.no_grad():
    for t, c in zip([0.0, 0.5 * tau, 0.95 * tau], [COL["dso"], COL["dgm"], COL["ask"]]):
        k = int(round(t / dt)); ex = [db_ex[k, jj], da_ex[k, jj - 1]]
        dso, dgm = dso_policy(t, Sq, qq).cpu().numpy(), dgm_policy(t, Sq, qq).cpu().numpy()
        for a, side in zip(ax, [0, 1]):
            a.plot(jj - Qbar, ex[side], color=c, label=f"t = {t:.2f} tau")
            a.plot(jj - Qbar, dgm[:, side], "--", color=c); a.plot(jj - Qbar, dso[:, side], "o", ms=3, color=c)
for a, s in zip(ax, ["bid spread  delta^b(t,q)", "ask spread  delta^a(t,q)"]):
    a.set_title(s, fontsize=10); a.set_xlabel("inventory q"); a.axhline(1 / kappa, color="gray", lw=0.5)
ax[0].legend(fontsize=8, title="exact: line,  DGM: dashed,  DSO: dots", title_fontsize=8); plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(hist_dso, lw=0.8, color=COL["dso"], label="DSO  J_hat(delta^theta)")
ax[0].axhline(v_ex[0, Qbar], color="k", ls="--", label="exact  J* = v(0, 0)")
ax[0].set_xlabel("iteration"); ax[0].legend(fontsize=8)
with torch.no_grad():
    vd = V(torch.zeros_like(qq), Sq, qq).cpu().numpy()
ax[1].plot(jj - Qbar, v_ex[0, jj], "k", label="exact"); ax[1].plot(jj - Qbar, vd, "--", color=COL["dgm"], label="DGM")
ax[1].set_xlabel("inventory q"); ax[1].set_title("value function v(0, S0, q)", fontsize=10); ax[1].legend(fontsize=8); plt.tight_layout()

In [ ]:
with torch.no_grad(): o = simulate(dso_policy, B=1, seed=7)                 # one session under the DSO quotes (cf. slide 9)
tt = np.arange(N + 1) * dt; S, Q = o["S"][0].cpu().numpy(), o["Q"][0].cpu().numpy()
d, dN = o["d"][0].cpu().numpy(), o["dN"][0].cpu().numpy(); bid, ask = S[:-1] - d[:, 0], S[:-1] + d[:, 1]
fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))
ax[0].plot(tt, S, color=COL["mid"], lw=0.8, label="baseline"); ax[0].plot(tt[:-1], bid, color=COL["bid"], lw=0.8, label="bid"); ax[0].plot(tt[:-1], ask, color=COL["ask"], lw=0.8, label="ask")
ax[0].plot(tt[:-1][dN[:, 0] > 0], bid[dN[:, 0] > 0], "^", color=COL["bid"], ms=4); ax[0].plot(tt[:-1][dN[:, 1] > 0], ask[dN[:, 1] > 0], "v", color=COL["ask"], ms=4)
Nb, Na = np.r_[0, dN[:, 0].cumsum()], np.r_[0, dN[:, 1].cumsum()]
ax[1].plot(tt, Nb, color=COL["bid"], label="N^b"); ax[1].plot(tt, Na, color=COL["ask"], label="N^a"); ax[1].plot(tt, Q, color="goldenrod", label="Q")
ax[0].legend(fontsize=8, ncol=3); ax[1].legend(fontsize=8, ncol=3); ax[1].set_xlabel("time (fraction of session)"); plt.tight_layout()